# Notebook 02: BMS 117 RAG Tutor Agent

A Retrieval-Augmented Generation tutor that answers questions about BMS 117 course
material using the OpenAI Agents SDK with ChromaDB via `chroma-mcp`.

**Prerequisite:** Run Notebook 01 first to populate the ChromaDB vector store.

## Modes
1. **Q&A** — Ask questions about lecture material, get cited answers
2. **Exam Explainer** — Explain why an answer is correct on the practice exam
3. **Quiz Generator** — Generate practice multiple-choice questions

## Setup

In [1]:
%pip install -q sentence-transformers chromadb chroma-mcp openai openai-agents python-dotenv mcp


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
import sys
import json
import warnings
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*resource_tracker.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

import transformers
transformers.logging.set_verbosity_error()

from openai import AsyncOpenAI
from agents import Agent, Runner, ModelSettings, function_tool, set_tracing_disabled
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio

set_tracing_disabled(True)

# OpenRouter setup (same pattern as Lecture 08 demos)
if os.environ.get("OPENROUTER_API_KEY"):
    agents_client = AsyncOpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    # AGENTS_MODEL = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=agents_client)
    # print("Using model: openai/gpt-4o-mini via OpenRouter")
    AGENTS_MODEL = OpenAIChatCompletionsModel(model="nvidia/nemotron-3-super-120b-a12b:free", openai_client=agents_client)
    print("Using model: nvidia/nemotron-3-super-120b-a12b:free via OpenRouter")
elif os.environ.get("OPENAI_API_KEY"):
    agents_client = AsyncOpenAI()
    AGENTS_MODEL = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=agents_client)
    print("Using model: gpt-4o-mini via OpenAI")
else:
    raise ValueError("Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env")

Using model: nvidia/nemotron-3-super-120b-a12b:free via OpenRouter


## Section 1: Connect to ChromaDB

Connect to the same ChromaDB instance populated by Notebook 01.
The agent uses `chroma-mcp` to search the `bms117_course_materials` collection.

In [25]:
# Point to the same ChromaDB directory from Notebook 01
CHROMA_DIR = str(Path("data/chroma_db").resolve())
COLLECTION_NAME = "bms117_course_materials"

chroma_cmd = str(Path(sys.executable).parent / "chroma-mcp")

chroma_mcp = MCPServerStdio(
    params={
        "command": chroma_cmd,
        "args": ["--client-type", "persistent", "--data-dir", CHROMA_DIR],
    },
    name="BMS 117 Course Materials",
)

print(f"ChromaDB: {CHROMA_DIR}")
print(f"Collection: {COLLECTION_NAME}")

ChromaDB: /workspaces/final-project-oscar-peng/data/chroma_db
Collection: bms117_course_materials


## Section 2: Load Exam Data

Load the parsed exam questions from Notebook 01 so the exam explainer
tool can look up answers directly.

In [26]:
EXAM_DATA_FILE = Path("data/exam_questions.json")

with open(EXAM_DATA_FILE) as f:
    exam_questions = json.load(f)

# Build a lookup dict by question number
exam_lookup = {q["num"]: q for q in exam_questions}

print(f"Loaded {len(exam_questions)} exam questions")

Loaded 51 exam questions


## Section 3: Define Tools

The tutor agent has three tools — following the `@function_tool` pattern from Lecture 08.
Each tool is a Python function that the agent calls when it decides it needs that capability.

In [27]:
@function_tool
def explain_exam_question(question_number: int) -> str:
    """Look up a practice exam question by number and return the correct answer,
    the answer text, and the mapped learning objective. Use this when a student
    asks about a specific practice exam question."""
    q = exam_lookup.get(question_number)
    if not q:
        return f"Question {question_number} not found. Valid range: 1-{len(exam_questions)}."
    
    return (
        f"Question {q['num']}: {q['stem']}\n\n"
        f"Correct Answer: ({q['correct_letter']}) {q['correct_text']}\n\n"
        f"Learning Objective: {q['objective']}"
    )


@function_tool
def list_exam_questions() -> str:
    """List all practice exam questions with their numbers. Use this when a student
    wants to browse available questions or doesn't know which number to ask about."""
    lines = []
    for q in exam_questions:
        lines.append(f"Q{q['num']}: {q['stem'][:80]}...")
    return "\n".join(lines)


# Verify tools are set up
print(f"Tool: {explain_exam_question.name}")
print(f"  → {explain_exam_question.description[:80]}")
print(f"Tool: {list_exam_questions.name}")
print(f"  → {list_exam_questions.description[:80]}")

Tool: explain_exam_question
  → Look up a practice exam question by number and return the correct answer,
the an
Tool: list_exam_questions
  → List all practice exam questions with their numbers. Use this when a student
wan


## Section 4: Build the Tutor Agent

The core RAG agent — it searches the vector store via `chroma-mcp` and uses
the exam tools when needed. The system prompt enforces grounded answers
with citations, matching the citing agent pattern from Lecture 08 Demo 2.

In [28]:
SYSTEM_PROMPT = """
You are an AI tutor for UCSF BMS 117 (Microbiology & Immunology for Dental Students).
You help students study for Exam 1 by answering questions, explaining concepts, and
generating practice questions.

RULES:
1. ALWAYS search the bms117_course_materials collection using chroma_query_documents
   before answering any content question. Use n_results=5 for thorough retrieval.
2. Answer based ONLY on retrieved course materials. Do not use outside knowledge.
3. ALWAYS cite your sources: include the lecture name and page numbers.
   Format: [Source: Lecture Name, pages X-Y]
4. When explaining exam questions, also state the relevant learning objective.
5. If you cannot find the answer in the course materials, say:
   "I cannot find this in the provided course documents."
6. When generating quiz questions, base them on retrieved lecture content and
   provide the correct answer with an explanation.

COLLECTION NAME: bms117_course_materials
"""

tutor_agent = Agent(
    name="BMS 117 Tutor",
    instructions=SYSTEM_PROMPT,
    mcp_servers=[chroma_mcp],
    tools=[explain_exam_question, list_exam_questions],
    model=AGENTS_MODEL,
    model_settings=ModelSettings(temperature=0),
)

print("✓ Tutor agent configured")
print(f"  MCP servers: {[s.name for s in tutor_agent.mcp_servers]}")
print(f"  Tools: {[t.name for t in tutor_agent.tools]}")

✓ Tutor agent configured
  MCP servers: ['BMS 117 Course Materials']
  Tools: ['explain_exam_question', 'list_exam_questions']


## Section 5: Test — Q&A Mode

Ask the tutor content questions about the lecture material.

In [29]:
from agents.items import ToolCallItem


async def ask_tutor(question: str) -> str:
    """Send a question to the tutor and display the response with tool call info."""
    async with chroma_mcp:
        result = await Runner.run(tutor_agent, input=question)
    
    # Show which tools the agent called
    tool_calls = [item for item in result.new_items if isinstance(item, ToolCallItem)]
    for tc in tool_calls:
        call = tc.raw_item
        print(f"  [Tool: {call.name}({call.arguments[:60]}...)]")
    
    print(f"\nQ: {question}\n")
    print(f"A: {result.final_output}")
    print("=" * 60)
    return result.final_output


# Test Q&A
_ = await ask_tutor("What are the mechanisms of horizontal gene transfer in bacteria?")

  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: What are the mechanisms of horizontal gene transfer in bacteria?

A: The three primary mechanisms of horizontal gene transfer (HGT) in bacteria are **transformation**, **transduction**, and **conjugation**.  

- **Transformation** – uptake of naked DNA from the environment by a naturally competent bacterium; the DNA can be incorporated into the genome via homologous recombination.  
- **Transduction** – transfer of bacterial DNA by a bacteriophage (virus) that packages host DNA instead of its own genome and delivers it to a new recipient cell.  
- **Conjugation** – direct cell‑to‑cell transfer of DNA (usually a plasmid) through a pilus; the donor cell transfers a copy of the plasmid to the recipient.  

These mechanisms allow rapid spread of traits such as antibiotic resistance and virulen

In [30]:
_ = await ask_tutor("How do dendritic cells activate T cells?")

  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: How do dendritic cells activate T cells?

A: Dendritic cells (DCs) activate T cells by capturing antigens in peripheral tissues, becoming mature, migrating to the T‑cell zones of lymph nodes, and then presenting processed peptide‑MHC complexes together with co‑stimulatory signals to naïve T cells. This two‑signal process (antigen presentation + co‑stimulation) leads to T‑cell activation and initiation of the adaptive immune response.

**Key steps supported by the course materials:**

1. **Antigen capture and DC maturation** – DCs encounter pathogens in tissues, become activated, and up‑regulate MHC and co‑stimulatory molecules.  
2. **Migration to lymph nodes** – Activated DCs travel via lymphatics to the T‑cell zone of secondary lymphoid organs.  
3. **Antigen presentation** – DCs display

In [31]:
_ = await ask_tutor("What is the difference between endotoxins and exotoxins?")

  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: What is the difference between endotoxins and exotoxins?

A: **Endotoxins vs. Exotoxins – Key Differences (based on BMS 117 course material)**  

| Feature | Endotoxin (LPS) | Exotoxin |
|---|---|---|
| **Chemical nature** | Lipopolysaccharide (LPS) – a lipid‑polysaccharide component of the outer membrane | Polypeptide proteins (secreted) |
| **Location** | Integral part of the cell wall of **Gram‑negative** bacteria; released mainly when the cell lyses | **Secreted** by both **Gram‑negative and Gram‑positive** bacteria |
| **Immunogenicity** | Poorly immunogenic; does not elicit strong protective immunity; no effective vaccines | Highly immunogenic; antitoxins and toxoid vaccines are effective (e.g., diphtheria, tetanus toxoids) |
| **Toxicity** | Low toxicity in small amounts; requires large quantities to cause systemic effects | High toxicity; even minute amounts can be potent |
| *

## Section 6: Test — Exam Explainer Mode

Ask the tutor to explain specific practice exam questions.

In [32]:
_ = await ask_tutor("Explain practice exam question 4. Why is the correct answer correct?")

  [Tool: explain_exam_question({"question_number": 4}...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: Explain practice exam question 4. Why is the correct answer correct?

A: **Why answer (c)is correct**

The practice exam question asks: *“With regard to bacterial genetics, ‘Transformation’ describes a process that involves:”*  
The correct choice is **(c) The uptake of free DNA in the environment by competent bacterial cells**.

**Explanation from the course materials**

- Transformation is one of the three major mechanisms of horizontal DNA transfer in bacteria (the others being transduction and conjugation).  
- In transformation, bacteria that are *competent* possess the molecular machinery to take up **naked (free) DNA** from their surroundings. Once inside the cell, this extracellular DNA can be incorporated into the recipien

In [33]:
_ = await ask_tutor("Explain why the answer to question 16 is IgE and not IgG.")

  [Tool: explain_exam_question({"question_number": 16}...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: Explain why the answer to question 16 is IgE and not IgG.

A: Thereaction described—sudden difficulty breathing and unconsciousness after a bee sting—is a classic example of systemic **anaphylaxis**, which occurs within minutes of re‑exposure to an antigen. This rapid, life‑threatening response is the hallmark of **Type I (immediate) hypersensitivity**, and it is mediated specifically by **IgE** antibodies, not IgG.

**Why IgE?**

* IgE is the antibody class that binds with high affinity to the FcεRI receptor on mast cells and basophils.  
* Upon re‑exposure to the antigen (bee venom), the antigen cross‑links the IgE molecules already bound to these cells, triggering massive degranulation and the release of pre‑formed mediators (h

## Section 7: Test — Quiz Generator Mode

Ask the tutor to generate practice questions on a topic.

In [34]:
_ = await ask_tutor(
    "Generate 3 multiple choice practice questions about bacterial genetics. "
    "Include answers and explanations."
)

  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]
  [Tool: chroma_query_documents({"collection_name": "bms117_course_materials", "query_texts"...)]

Q: Generate 3 multiple choice practice questions about bacterial genetics. Include answers and explanations.

A: **Bacterial Genetics – Practice Multiple‑Choice Questions**  
*(All answers and explanations are based solely on the retrieved UCSF BMS 117 course materials.)*

---

### Question 1  
**Which statement best describes the difference between the bacterial chromosome and a plasmid?**  

A. Chromosome carries virulence factors; plasmid carries essential metabolic genes.  
B. Chromosome is linear; plasmid is always circular.  
C. Chromosome contains genes necessary for survival and adaptation; plasmid carries genes that confer advantages such as antibiotic resistance.  
D. Plasmid replicates

## Section 8: RAG vs Direct LLM Comparison

Compare the RAG tutor against a base LLM without retrieval — the same
comparison from Lecture 08 Demo 2, Section 4. This demonstrates the value
of grounding answers in course materials.

In [35]:
# Direct LLM (no retrieval, no tools)
direct_agent = Agent(
    name="Direct LLM",
    instructions="You are a helpful tutor. Answer concisely.",
    model=AGENTS_MODEL,
    model_settings=ModelSettings(temperature=0),
)

test_question = (
    "According to BMS 117 lecture materials, what is the role of AIRE "
    "in thymic epithelial cells and why is it important for immune tolerance?"
)

# RAG response
async with chroma_mcp:
    rag_result = await Runner.run(tutor_agent, input=test_question)

# Direct LLM response
direct_result = await Runner.run(direct_agent, input=test_question)

print("RAG TUTOR (grounded in course materials):")
print(rag_result.final_output)
print("\n" + "-" * 60 + "\n")
print("DIRECT LLM (from training data only):")
print(direct_result.final_output)

RAG TUTOR (grounded in course materials):
AIRE (AutoImmune REgulator) is a transcription factor expressed in medullary thymic epithelial cells (mTECs). Its role is to drive the expression of a wide array of tissue‑restricted self‑antigens (TRAs) – proteins that are normally found only in specific peripheral tissues. These self‑proteins are then processed and presented on MHC molecules to developing thymocytes.  

By presenting these peripheral tissue antigens in the thymus, AIRE enables **negative selection** (central tolerance): thymocytes that bind strongly to these self‑peptide/MHC complexes receive apoptotic signals and are deleted. This removes self‑reactive T‑cell clones before they can enter the periphery, thereby preventing autoimmune attack on the body’s own tissues.  

If AIRE is defective, the thymus fails to express many TRAs, autoreactive T cells escape deletion, and multi‑organ autoimmunity develops – exemplified by Autoimmune Polyendocrinopathy Syndrome type 1 (APS‑1, al

## Next Steps

Proceed to **Notebook 03** to run the full evaluation:
all 51 practice exam questions through both RAG and direct LLM, compared
against the answer key.